<a href="https://colab.research.google.com/github/Emelygabrielaa/calgary-building-permits-eda/blob/main/notebooks/02_q1_processing_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project: Calgary Building Permits, Descriptive Analysis and EDA**

**Path:** Statistician path, Project 1 of 11  
**Question:** 1 of 5, Processing time (how long do applications take?)  
**Author:** Emely Rodriguez  
**Languages:** Python (lead), R (cross-check in a separate notebook)  
**Data as of:** 2026-10-27 snapshot, processed dataset (502,814 rows, 46 columns).   
**Licence:** Contains information licensed under the Open Government Licence - City of Calgary.

## Project phases

| Phase | Content | Status |
|---|---|---|
| 0. Data audit | Structure, quality and definitions of the source data | Done |
| 1. Analysis dataset | Handling of audit issues, selected columns, derived variables, dated snapshot | Done |
| **2. Analysis** | **Part A (descriptive) and Part B (formal tests and models), one notebook per question** | **This notebook (Question 1)** |
| 3. Report | Findings and conclusions | Pending |

## What this project is about
The City of Calgary publishes its building permit application records, which start in June 1999. In this project I use that table to answer five practical questions: how long applications take, how much the projects are worth, how activity changes over time, where it happens, and how complete the data is. In each question's notebook, Part A answers it with descriptive statistics and Part B follows with formal tests and models.

## The five questions

| # | Topic | Question | Part A: descriptive | Part B: going deeper |
|---|---|---|---|---|
| **1** | **Processing time** | **How long do applications take?** | **Days from application to issue, with percentiles by permit type** | **Kruskal-Wallis with pairwise tests, and Kaplan-Meier curves** |
| 2 | Project cost | How much are the projects worth? | Skewness, log scale, tail, and summaries by class | Lognormal fit check, bootstrap intervals for medians, quantile regression |
| 3 | Volume over time | How does activity change over time? | Counts by year and month, seasonality plots | Decomposition, Mann-Kendall trend test, count regression |
| 4 | Geography | Where does it happen? | Counts and medians by community, and a map | Rates per community and Moran's I |
| 5 | Missingness | How complete is the data? | Missing rates by column and permit type | Chi-square tests and logistic regression of missingness |

This notebook covers Question 1 (in bold). The other questions are in their own notebooks.

## The data

- **Source:** City of Calgary Open Data portal, dataset "Building Permits" (ID c2es-76ed)
- **Provider:** The City of Calgary, Dev Business and Bldg Srvc
- **What a row is:** a building permit application record, identified by `permitnum` (unique, checked in section 0.2.1 of the Phase 0 notebook)
- **Coverage:** applications from 1999-06-22 onward, updated daily
- **Size:** about 503,000 rows and 30 columns in the portal. The processed dataset used here has 502,814 rows and 46 columns, built in Phase 1 from a snapshot of 21 columns.
- **File used:** `data/processed/building_permits_processed_2026-10-27.parquet` in the project repository (Phase 1)
- **Coordinates:** WGS84 (EPSG:4326), in degrees
- **Licence:** City of Calgary Open Data Terms (https://data.calgary.ca/d/Open-Data-Terms/u45n-7awa)


**Reproducibility note:** this notebook reads the processed dataset built in Phase 1, which is fixed, so the values do not change. The dataset was built from the portal snapshot of 2026-10-27. The handling decisions and the definitions of the variables are in the Phase 1 notebook, and this notebook does not query the portal.

## Pills for this project
**Board pills:** Descriptive/EDA, Group Comparison, Missing-data analysis, Trend analysis,
Regression analysis, Uncertainty analysis, Spatial

**Method pills:** percentiles, ecdf, skewness, log-transform, heavy-tails, kruskal-wallis,
multiple-comparisons, censoring, kaplan-meier, bootstrap, quantile-regression, seasonality,
decomposition, mann-kendall, count-regression, rates-per-area, morans-i, choropleth,
missing-rates, chi-square, logistic-regression, mar-vs-mcar

----------

# **Phase 2, Question 1: Processing time**

**Question:** how long do applications take?

**Goal:** describe the time from application to issue (Part A), and then test how it differs between permit types and how it behaves for applications that have not been issued yet (Part B).

**Input:** the processed dataset built in Phase 1. The handling decisions and the definitions of the variables are in the Phase 1 notebook, and this notebook does not repeat them.

## Q1.0 Setup

This notebook has its own setup, so it can run without the other notebooks.

### Q1.0.1 Libraries and data address

In [1]:
# Import the libraries: web requests, file handling, and data tables
import io, requests, pandas as pd

# Date of the snapshot the processed dataset was built from
SNAP_DATE = "2026-10-27"

# Address of the processed dataset in the public repository (raw file)
PROCESSED_URL = ("https://raw.githubusercontent.com/Emelygabrielaa/"
                 "calgary-building-permits-eda/main/data/processed/"
                 f"building_permits_processed_{SNAP_DATE}.parquet")

### Q1.0.2 Loading the processed dataset

In [2]:
# Download the processed dataset and read it as a table
r = requests.get(PROCESSED_URL, timeout=300)
r.raise_for_status()                                   # stop with an error if the download fails
df = pd.read_parquet(io.BytesIO(r.content))

# Check the size: it must be 502,814 rows and 46 columns
print(df.shape)
print("Unique permit numbers:", df["permitnum"].nunique())

(502814, 46)
Unique permit numbers: 502814


The processed dataset has 502,814 rows and 46 columns, with one row for each permit number, as built in Phase 1.